In [1]:
import sys
from pathlib import Path

# Get project root - the project root should have both 'vault' and 'ensemble' directories
# Walk up from current directory until we find a directory with both
current_dir = Path.cwd()
project_root = current_dir

# Walk up the directory tree to find project root
while project_root != project_root.parent:
    # Check if this directory has both vault and ensemble (indicating project root)
    has_vault = (project_root / 'vault').exists()
    has_ensemble = (project_root / 'ensemble').exists()
    
    if has_vault and has_ensemble:
        # Found the project root
        break
    
    project_root = project_root.parent
else:
    # If we reached the filesystem root without finding it, use parent of current_dir
    # (assuming we're in research/ subdirectory)
    if (current_dir.parent / 'vault').exists() and (current_dir.parent / 'ensemble').exists():
        project_root = current_dir.parent
    else:
        # Last resort: use current directory
        project_root = current_dir

# Add project root to Python path
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Current directory: {current_dir}")
print(f"Project root: {project_root}")
print(f"Has vault: {(project_root / 'vault').exists()}")
print(f"Has ensemble: {(project_root / 'ensemble').exists()}")
print(f"Python path includes: {str(project_root) in sys.path}")

Current directory: /home/raman/repos/Trading-Algo/research
Project root: /home/raman/repos/Trading-Algo
Has vault: True
Has ensemble: True
Python path includes: True


In [2]:
"""
Portfolio Testing with PortfolioTester

This notebook demonstrates how to use the PortfolioTester framework to:
1. Load data and create portfolios
2. Fit portfolios using candles
3. Generate predictions at different granularity levels
4. Calculate returns and generate tearsheets
5. Analyze performance at portfolio, ensemble, and base model levels
"""

from datetime import datetime
import pandas as pd
import numpy as np

from utils.enums import TimeFrame, Ticker
from utils import helpers
from ensemble.portfolio import Portfolio
from ensemble.diversified_ensemble import DiversifiedEnsemble
from ensemble.portfolio_tester import PortfolioTester

# Portfolio Testing Framework Demo

## Overview

The `PortfolioTester` class provides a comprehensive framework for testing portfolios:
- **Automatic returns calculation**: Calculates log returns from candles automatically
- **Granular predictions**: Get predictions at portfolio, ensemble, and base model levels
- **Integrated tearsheets**: Generate QuantStats tearsheets at all granularity levels
- **Baseline comparison**: Compare against equal-weight or buy-and-hold baselines

In [3]:
# Load data
print("Loading data...")



tickers = [ Ticker.ES, Ticker.NQ, Ticker.YM, Ticker.RTY]


train_candles = helpers.load_data_multi_ticker(
    tickers=tickers,
    timeframe=TimeFrame.D,
    start=datetime(2000, 1, 1),
    end=datetime(2020, 1, 1),
    use_millisecond_offset=True
)

test_candles = helpers.load_data_multi_ticker(
    tickers=tickers,
    timeframe=TimeFrame.D,
    start=datetime(2020, 1, 1),
    end=datetime(2024, 1, 1),
    use_millisecond_offset=True
)

print(f"Training data: {len(train_candles)} candles")
print(f"Test data: {len(test_candles)} candles")
print(f"Tickers: {sorted(train_candles['ticker'].unique())}")

Loading data...
Training data: 15258 candles
Test data: 4032 candles
Tickers: [<Ticker.ES: 'US500'>, <Ticker.NQ: 'US100'>, <Ticker.YM: 'US30'>, <Ticker.RTY: 'RUSSELL_2000'>]


## Step 1: Create Portfolio

For this demo, we'll use a simple buy-and-hold ensemble. In practice, you would load your ensemble from the vault or create it from feature configs.

In [4]:
# Load ensemble from vault using vault_manager
from ensemble.vault_manager import load_ensemble_from_vault

# Load ensemble from vault
target_vol = 0.15
ensemble_dir = 'vault/D/buy_hold_long'
print(f"Loading ensemble from {ensemble_dir}...")
ensemble = load_ensemble_from_vault(ensemble_dir, refit=True, target_volatility=target_vol)
rsi_ensemble = load_ensemble_from_vault('vault/D/rsi_bias_lookback_2_long', refit=True, target_volatility=target_vol)
momentum_ensemble = load_ensemble_from_vault('vault/D/indices_momentum_long', refit=True, target_volatility=target_vol)
rsi_regime_ensemble = load_ensemble_from_vault('vault/D/rsi_regime_long', refit=True, target_volatility=target_vol)
rsi_regime_short_ensemble = load_ensemble_from_vault('vault/D/rsi_regime_short', refit=True, target_volatility=target_vol)
cum_sum_rsi_ensemble = load_ensemble_from_vault('vault/D/cum_rsi_2_long', refit=True, target_volatility=target_vol)
rsi_5_long_ensemble = load_ensemble_from_vault('vault/D/rsi_5_long', refit=True, target_volatility=target_vol)
ultimate_c_ensemble = load_ensemble_from_vault('vault/D/ultimate_c_2_3_3 _long', refit=True, target_volatility=target_vol)
print(f"Ensemble loaded: {ensemble}")
print(f"Ensemble fitted: {ensemble.is_fitted_}")
print(f"Base models: {list(ensemble.base_models.keys())}")

# Create portfolio
portfolio = Portfolio(
    ensembles=[ rsi_regime_ensemble, momentum_ensemble, rsi_ensemble, ensemble, rsi_regime_short_ensemble, cum_sum_rsi_ensemble, rsi_5_long_ensemble, ultimate_c_ensemble],
    trading_timeframe=TimeFrame.D,
    target_volatility=target_vol,
    max_position_pct=2.5
)
print(f"\nPortfolio created with {len(portfolio.ensembles)} ensemble(s)")

Loading ensemble from vault/D/buy_hold_long...
Ensemble loaded: DiversifiedEnsemble (not fitted)
  Target Volatility: 0.15
Ensemble fitted: False
Base models: ['buy_hold_signal_D::quantile_binning_2']

Portfolio created with 8 ensemble(s)


## Step 2: Initialize PortfolioTester

The PortfolioTester wraps your portfolio and provides a convenient API for testing.

In [5]:
# Initialize PortfolioTester
tester = PortfolioTester(
    portfolio=portfolio,
    baseline_mode='equal_weight'  # or 'buy_hold'
)

# Fit portfolio (automatically calculates log returns)
print("\nFitting portfolio...")
tester.fit(train_candles)

# Print comprehensive diagnostics
tester.portfolio.print_diagnostics()

# Also print individual values for easy access
diagnostics = tester.portfolio.get_diagnostics()
print(f"\n📋 Quick Summary:")
print(f"  IDM: {diagnostics.get('idm', 'Not calculated')}")
print(f"  FDM: {diagnostics.get('fdm', 'Not calculated')}")
print(f"  Mean Return Correlation: {diagnostics.get('mean_return_correlation', 'N/A')}")
print(f"  Mean Forecast Correlation: {diagnostics.get('mean_forecast_correlation', 'N/A')}")
print(f"  Ensemble fitted: {ensemble.is_fitted_}")


Fitting portfolio...
⚡ Fitting 8 ensembles in parallel (using joblib)...


BaseModel.predict() has 3348 NaN values out of 5063. This indicates bias nodes are not processing all candles correctly.
BaseModel.predict() has 1715 NaN values out of 5063. This indicates bias nodes are not processing all candles correctly.
BaseModel.predict() has 3290 NaN values out of 4479. This indicates bias nodes are not processing all candles correctly.
BaseModel.predict() has 364 NaN values out of 652. This indicates bias nodes are not processing all candles correctly.
BaseModel.predict() has 1189 NaN values out of 4479. This indicates bias nodes are not processing all candles correctly.
BaseModel.predict() has 288 NaN values out of 652. This indicates bias nodes are not processing all candles correctly.


✓ Parallel fitting completed in 89.97s (0 failed)

Fitting WeightLayer...


Found constant forecast columns (all same value): ['buy_hold_signal_D::quantile_binning_2']. These will be treated as perfectly correlated (corr=1.0) for FDM calculation.
Found constant forecast columns (all same value): ['buy_hold_signal_D::quantile_binning_2']. These will be treated as perfectly correlated (corr=1.0) for FDM calculation.
Found constant forecast columns (all same value): ['buy_hold_signal_D::quantile_binning_2']. These will be treated as perfectly correlated (corr=1.0) for FDM calculation.
Found constant forecast columns (all same value): ['buy_hold_signal_D::quantile_binning_2']. These will be treated as perfectly correlated (corr=1.0) for FDM calculation.



✓ WeightLayer fitted successfully!
  Tickers: 4
  Mean FDM: 1.4656 (range: 1.4581 - 1.4809)
  Mean models per ticker: 8.0
  Per-ticker FDM:
    Ticker.ES: FDM=1.4581 (8 model(s))
    Ticker.NQ: FDM=1.4581 (8 model(s))
    Ticker.YM: FDM=1.4655 (8 model(s))
    Ticker.RTY: FDM=1.4809 (8 model(s))

PORTFOLIO DIAGNOSTICS

📊 Portfolio Status:
  Fitted: True
  Trading Timeframe: D
  Number of Instruments: 4
  Instruments: ['ES', 'NQ', 'YM', 'RTY']

🎯 Instrument Diversification Multiplier (IDM):
  IDM: 1.074179832225252
  IDM Max: 2.5
  Mean Return Correlation: 0.856654530298174

🔮 Forecast Diversification Multiplier (FDM):
  Mean FDM: 1.4656448544764245
  FDM Range: 1.4580845134757265 - 1.4809281344717182
  FDM Max: 2.5
  Number of Tickers: 4
  Mean Models per Ticker: 8.0

⚖️  Weight Layer:
  Fitted: True
  Weight Method: inverse_correlation

  Per-Ticker FDM and Models:
    Ticker.ES:
      FDM: 1.4581
      Models: 8 (buy_hold_signal_D::quantile_binning_2, cumulative_rsi_signal_D_avgPeri

## Step 3: Generate Predictions

You can get predictions at different granularity levels:
- **Portfolio level**: Aggregated positions (default)
- **Ensemble level**: Individual ensemble predictions
- **Base model level**: Individual base model predictions

In [6]:
# Basic prediction (portfolio level only)
print("\nGenerating portfolio-level predictions...")
positions = tester.predict(test_candles)
print(f"Generated {len(positions)} position predictions")
print(f"Position fraction range: [{positions['position_fraction'].min():.4f}, {positions['position_fraction'].max():.4f}]")



Generating portfolio-level predictions...
⚡ Getting predictions from 8 ensembles in parallel (using joblib)...


BaseModel.predict() has 295 NaN values out of 1008. This indicates bias nodes are not processing all candles correctly.
BaseModel.predict() has 713 NaN values out of 1008. This indicates bias nodes are not processing all candles correctly.


✓ Parallel prediction completed in 9.67s
Generated 4032 position predictions
Position fraction range: [0.0350, 0.5371]


In [7]:

# With ensemble-level and base model-level granularity
print("\nGenerating granular predictions (ensemble + base model level)...")
results = tester.predict(
    test_candles,
    return_ensemble_predictions=True,
    return_base_model_predictions=True
)

portfolio_positions = results['portfolio']
ensemble_predictions = results['ensembles']
base_model_predictions = results['base_models']

print(f"\nGranular predictions:")
print(f"  Portfolio positions: {len(portfolio_positions)}")
print(f"  Ensemble predictions: {len(ensemble_predictions)}")
print(f"  Base model predictions: {len(base_model_predictions)}")


Generating granular predictions (ensemble + base model level)...
🚀 Portfolio.predict_from_candles() cache HIT for date_range: (datetime.date(2020, 1, 2), datetime.date(2023, 12, 29)) (hits: 1, misses: 1)

Granular predictions:
  Portfolio positions: 4032
  Ensemble predictions: 8
  Base model predictions: 8


In [8]:
portfolio_positions

,ticker,datetime,forecast_score,position_fraction
0,Ticker.ES,2020-01-02 00:00:00.000,0.724411,0.194537
1,Ticker.NQ,2020-01-02 00:00:00.001,0.555360,0.149139
2,Ticker.YM,2020-01-02 00:00:00.002,0.856771,0.230082
3,Ticker.RTY,2020-01-02 00:00:00.003,0.276914,0.074364
4,Ticker.ES,2020-01-03 00:00:00.000,0.724411,0.194537
...,...,...,...,...
4027,Ticker.RTY,2023-12-28 00:00:00.003,0.276914,0.074364
4028,Ticker.ES,2023-12-29 00:00:00.000,1.112867,0.298855
4029,Ticker.NQ,2023-12-29 00:00:00.001,0.943815,0.253457
4030,Ticker.YM,2023-12-29 00:00:00.002,1.248297,0.335224


## Step 4: Calculate Returns

Calculate strategy returns and baseline returns for comparison.

In [9]:
# Calculate strategy returns
print("\nCalculating strategy returns...")
strategy_returns = tester.calculate_strategy_returns(test_candles)
print(f"Strategy returns: {len(strategy_returns)} days")
print(f"Total return: {strategy_returns.sum():.4f}")
if strategy_returns.std() > 0:
    sharpe = strategy_returns.mean() / strategy_returns.std() * np.sqrt(252)
    print(f"Annualized Sharpe: {sharpe:.4f}")

# Calculate baseline returns
print("\nCalculating baseline returns...")
baseline_returns = tester.calculate_baseline_returns(test_candles)
print(f"Baseline returns: {len(baseline_returns)} days")
print(f"Total return: {baseline_returns.sum():.4f}")
if baseline_returns.std() > 0:
    baseline_sharpe = baseline_returns.mean() / baseline_returns.std() * np.sqrt(252)
    print(f"Annualized Sharpe: {baseline_sharpe:.4f}")

# Compare performance
print(f"\nPerformance Comparison:")
print(f"  Strategy total return: {strategy_returns.sum():.4f} ({strategy_returns.sum()*100:.2f}%)")
print(f"  Baseline total return: {baseline_returns.sum():.4f} ({baseline_returns.sum()*100:.2f}%)")
print(f"  Excess return: {(strategy_returns.sum() - baseline_returns.sum()):.4f} ({(strategy_returns.sum() - baseline_returns.sum())*100:.2f}%)")


Calculating strategy returns...
Strategy returns: 4028 days
Total return: 0.6773
Annualized Sharpe: 0.7181

Calculating baseline returns...
Baseline returns: 4028 days
Total return: 1.5205
Annualized Sharpe: 0.3724

Performance Comparison:
  Strategy total return: 0.6773 (67.73%)
  Baseline total return: 1.5205 (152.05%)
  Excess return: -0.8432 (-84.32%)


## Step 5: Generate Tearsheets

Generate QuantStats tearsheets for performance analysis.

In [10]:
# Generate portfolio-level tearsheet
print("\nGenerating portfolio tearsheet...")
tester.generate_tearsheet(
    strategy_name='Buy Hold Portfolio',
    output_dir='reports/portfolio',  # Save HTML report to this directory
    mode='html',  # 'html', 'full', 'basic', or 'metrics'
    candles_df=test_candles
)

# Generate ensemble-level tearsheets (if available)
if tester.ensemble_predictions:
    print("\nGenerating ensemble tearsheets...")
    tester.generate_ensemble_tearsheets(
        output_dir='reports/ensembles',  # Save HTML reports to this directory
        mode='html',
        candles_df=test_candles
    )

# Generate base model-level tearsheets (if available)
if tester.base_model_predictions:
    print("\nGenerating base model tearsheets...")
    tester.generate_base_model_tearsheets(
        output_dir='reports/base_models',  # Save HTML reports to this directory
        mode='html',
        candles_df=test_candles
    )


Generating portfolio tearsheet...


findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font f


✅ HTML tearsheet saved to: reports/portfolio/Buy_Hold_Portfolio_tearsheet.html
   Note: Using non-compounded returns (Robert Carver methodology)

Generating ensemble tearsheets...


findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font f


✅ HTML tearsheet saved to: reports/ensembles/ensemble_0_tearsheet.html
   Note: Using non-compounded returns (Robert Carver methodology)


...


✅ HTML tearsheet saved to: reports/base_models/ensemble_5_cumulative_rsi_signal_D_avgPeriod_[2]_lookback_[2]_quantile_binning_10_tearsheet.html
   Note: Using non-compounded returns (Robert Carver methodology)


findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font f


✅ HTML tearsheet saved to: reports/base_models/ensemble_6_rsi_signal_D_lookback_[5]_quantile_binning_10_tearsheet.html
   Note: Using non-compounded returns (Robert Carver methodology)


findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font f


✅ HTML tearsheet saved to: reports/base_models/ensemble_7_ultimate_c_signal_D_factor_[3]_lookback_[2]_smoothLookback_[3]_quantile_binning_10_tearsheet.html
   Note: Using non-compounded returns (Robert Carver methodology)
